<a href="https://colab.research.google.com/github/Lumthrong/Whisper-API/blob/main/Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install openai-whisper fastapi uvicorn python-multipart pyngrok
!apt-get update && apt-get install -y ffmpeg

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [6]:

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import whisper
import tempfile
import os
import uuid
import threading

# ================= APP =================
app = FastAPI()

# ================= MODEL =================
print("🔄 Loading Whisper model...")
model = whisper.load_model("base")  # load once globally
print("✅ Model loaded")

# ================= JOB STORE =================
jobs = {}

# ================= ROUTES =================
@app.get("/")
def home():
    return {"status": "ok"}

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": True
    }

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):

    job_id = str(uuid.uuid4())

    suffix = os.path.splitext(file.filename)[1] or ".mp3"

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp:
        content = await file.read()

        # 🚨 Skip tiny/broken files early
        if len(content) < 2000:
            return JSONResponse(content={
                "jobId": job_id,
                "status": "completed",
                "segments": []
            })

        temp.write(content)
        temp_path = temp.name

    jobs[job_id] = {"status": "processing"}

    # ================= BACKGROUND PROCESS =================
    def process():
        try:
            print("🚀 Processing started:", job_id)

            # ✅ Transcribe
            result = model.transcribe(temp_path)

            # 🚨 Validate result
            if not result or "segments" not in result:
                raise Exception("Invalid result")

            segments_list = [
                {
                    "start": seg.get("start", 0),
                    "text": seg.get("text", "").strip()
                }
                for seg in result["segments"]
                if seg.get("text", "").strip()
            ]

            print("🧠 Segments count:", len(segments_list))

            # 🚨 Handle silence safely
            if not segments_list:
                print("⚠️ No speech detected")
                jobs[job_id] = {
                    "status": "completed",
                    "segments": []
                }
                return

            jobs[job_id] = {
                "status": "completed",
                "segments": segments_list
            }

            print("✅ Processing completed:", job_id)

        except Exception as e:
            print("⚠️ Skipping bad chunk:", e)

            jobs[job_id] = {
                "status": "completed",
                "segments": []
            }

        finally:
            if os.path.exists(temp_path):
                os.remove(temp_path)

    # 🔥 SIMPLE THREAD (Colab safe)
    threading.Thread(target=process).start()

    return JSONResponse(content={"jobId": job_id})

@app.get("/status/{job_id}")
def get_status(job_id: str):
    return jobs.get(job_id, {"status": "failed"})

🔄 Loading Whisper model...
✅ Model loaded


In [7]:
from pyngrok import ngrok

ngrok.set_auth_token("3BWf4XlGDP1I26xpT2chBKb3ggB_4y76YVLz1Sur5BrLVzvVh")

In [8]:

import threading

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

public_url = ngrok.connect(8000)
print("🔥 YOUR API:", public_url)

Exception in thread Thread-6 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_4124/1967584689.py", line 4, in run
NameError: name 'uvicorn' is not defined


🔥 YOUR API: NgrokTunnel: "https://marcell-uncaused-nonproblematically.ngrok-free.dev" -> "http://localhost:8000"
